Generalized Additive Models:

Contributions: 
1. https://web.stanford.edu/class/stats202/notes/Nonlinear/GAM.html
2. https://academic.oup.com/jrsssc/article/64/1/139/7067572#397191485
3. https://medium.com/just-another-data-scientist/building-interpretable-models-with-generalized-additive-models-in-python-c4404eaf5515
4. https://ecogambler.netlify.app/blog/autocorrelated-gams/
5. https://www.geeksforgeeks.org/artificial-intelligence/generalized-additive-model-in-python/

In [ ]:
# Import libraries:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pygam import LogisticGAM
#from pygam import GAM, s, f
import os

In [1]:
pip install pygam 

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement pygam (from versions: none)
ERROR: No matching distribution found for pygam


In [ ]:
# Data import and pre-processing:

os.chdir('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/data_input')

# Data import:
df_factor = pd.read_csv("value_labeled.csv", parse_dates=['Date'])
df_factor = df_factor[['Date','trend_bin']]    # trend_bin = trend label
df_factor = df_factor.set_index('Date')

df_features = pd.read_csv("final_features.csv", parse_dates=['Date'])
df_features = df_features.set_index('Date')


data = pd.concat([df_features,df_factor], axis=1, join='inner')
data = data.dropna()

print(data.columns)

Index(['Momentum_vs_Value_trend_lag1', 'Momentum_relative_volatility_lag1',
       'SA_NB_Curvature_lag1', 'Momentum_vs_Quality_lag1',
       'SA_NB_Level_TS_lag1', 'Value_vs_Quality_trend_lag1',
       'SA_NB_Curvature_QS_lag1', 'Value_excess_skewness_lag1',
       'Value_excess_kurtosis_lag1', 'US_NB_Slope_QS_lag1',
       'Momentum_relative_kurtosis_lag1', 'Value_vs_Quality_Cumulative_lag1',
       'US_NB_Level_TS_lag1', 'SA_NB_Slope_TS_lag1', 'SA_NB_Slope_QS_lag1',
       'Momentum_excess_skewness_lag1', 'Momentum_excess_momentum_lag1',
       'Quality_momentum_12M_lag1', 'Quality_relative_volatility_lag1',
       'Momentum_vs_Quality_trend_lag1', 'Quality_excess_momentum_lag1',
       'Momentum_momentum_1M_lag1', 'SA_NB_Level_QS_lag1',
       'Quality_momentum_24M_lag1', 'Momentum_lag1',
       'Momentum_excess_kurtosis_lag1', 'Value_relative_volatility_lag1',
       'US_NB_Slope_TS_lag1', 'US_NB_Level_lag1',
       'Quality_relative_skewness_lag1', 'US_NB_Curvature_lag1',
       

In [9]:
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values
feature_names = data.columns[:-1].tolist()  # Get feature names for later use
X = np.nan_to_num(X, nan=np.nan, posinf=1e10, neginf=-1e10)
print(X)
print(y)
    

[[ 0.          0.66143783 -1.00441809 ...  0.213      -6.60645161
   0.37      ]
 [ 0.          0.66143783 -1.00441809 ...  0.213      -6.60645161
   0.37      ]
 [ 0.          0.66143783 -1.00441809 ...  0.116      -6.60645161
   0.37      ]
 ...
 [ 1.          0.77643293 -7.31732581 ...  0.972       0.49758153
   0.09      ]
 [ 0.          0.71086373 -7.383612   ...  0.972       0.77825104
  -0.05      ]
 [ 0.          1.13378573 -7.6400072  ...  0.972       0.89111312
  -0.08      ]]
[1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 0. 0. 0. 0. 0. 1. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 1. 1. 0. 1. 0. 0. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0.
 0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 

In [ ]:

gam = GAM(s(0, n_splines=5) + s(1) + f(2) + s(3), distribution=’gamma’, link=’log’)